# QLoRA 4-bit 微調實戰（LLaMA-3 + NF4 + SFTTrainer）

## 學習目標

本 notebook 示範如何對大型語言模型進行 **QLoRA（Quantized LoRA）** 微調：

1. 用 `BitsAndBytesConfig` 以 4-bit NF4 量化載入模型（2026 標準寫法）
2. 理解 4-bit / 8-bit / 16-bit 何時用、`nf4` vs `fp4`、double quantization 成本效益
3. 正確的 PEFT 前置順序：量化 → `prepare_model_for_kbit_training()` → LoRA
4. 以 `tokenizer.apply_chat_template()` 取代硬寫的 `Human:/Assistant:` 格式
5. 以 `trl.SFTTrainer` 取代手刻 `Trainer` + 手刻 `-100` 標籤遮罩
6. 對照「底層手刻版」理解 response-only 標籤遮罩原理
7. 使用 `merge_and_unload()` 合併 adapter 並推送至 HuggingFace Hub

## 前置知識

- 已完成 `04-kbits-tuning/04-1bits_training/` 的 LoRA 基礎實驗
- 了解 LoRA 原理（低秩分解、`r`、`alpha`、`target_modules`）
- 了解 Transformer causal LM 的訓練迴路

## 相鄰 notebook

- 上一個：[../04-3bits_training/llama2_lora_3bit.ipynb](../04-3bits_training/llama2_lora_3bit.ipynb)
- 下一個：[../../05-Multimodal/](../../05-Multimodal/) — processor 抽象升級

## VRAM 需求（參考）

| 模型 | 量化 | LoRA rank | 近似 VRAM |
|------|------|-----------|----------|
| LLaMA-3-8B | 4-bit NF4 | r=8 | ~6 GB |
| LLaMA-3-70B | 4-bit NF4 | r=8 | ~40 GB |
| LLaMA-2-13B | 4-bit NF4 | r=8 | ~8 GB |

In [ ]:
# Cell 0 — 版本鎖定（所有後續 cell 依賴此處定義的版本）
# 執行前請確認環境符合以下版本，或在新環境中 pip install 這些套件

# pip install \
#   "transformers>=4.46" \
#   "datasets>=3.0" \
#   "trl>=0.12" \
#   "peft>=0.13" \
#   "accelerate>=1.0" \
#   "bitsandbytes>=0.44" \
#   "evaluate>=0.4" \
#   "safetensors>=0.4" \
#   "torch>=2.4"

import transformers, datasets, trl, peft, accelerate, bitsandbytes, torch

print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"BF16 support : {torch.cuda.is_bf16_supported()}")

## Step 1  匯入套件

2026 版使用以下核心元件：

- `BitsAndBytesConfig`：統一管理 4-bit 量化設定
- `prepare_model_for_kbit_training`：在注入 LoRA 前凍結量化層、轉換 LayerNorm 精度
- `trl.SFTTrainer` 與 `trl.SFTConfig`：負責 response-only 標籤遮罩與格式化，不需手刻 `-100`
- `set_seed`：保證訓練可重現

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import set_seed

set_seed(42)

## Step 2  載入資料集

從 HuggingFace Hub 載入 `alpaca_data_zh`，無本機路徑依賴，任何環境都能直接執行。

`alpaca_data_zh` 是一份繁／簡中文指令資料集，每筆包含：
- `instruction`：任務描述
- `input`：補充輸入（可為空字串）
- `output`：期望輸出

In [ ]:
ds = load_dataset("silk-road/alpaca-data-gpt4-chinese", split="train")
print(ds)
print(ds[0])

In [ ]:
# 查看前三筆確認欄位格式
for i in range(3):
    print(f"--- sample {i} ---")
    print("instruction:", ds[i]["instruction"][:80])
    print("input      :", ds[i]["input"][:40])
    print("output     :", ds[i]["output"][:80])
    print()

## Step 3  載入 Tokenizer

### 為何 LLaMA 的 `padding_side` 要設為 `right`？

Causal LM 訓練時，labels 是 input_ids 往右移一格，左側的 padding token 若不對齊會讓 loss 計算在錯誤位置累加。  
`padding_side='right'` 讓所有序列從左對齊，padding 補在右側，與 `-100` 標籤遮罩結合才能正確屏蔽 padding token 的 loss。

### 為何設 `pad_token`？

LLaMA 系列原始 tokenizer 沒有 `pad_token`。不設置的話，`DataCollator` 或 `SFTTrainer` 在做 padding 時會出錯。  
常見做法是把 `pad_token` 設為 `eos_token`（或保留 token id 2），推論時再用 `attention_mask` 告訴模型哪些位置是 padding。

In [ ]:
# Configure model id via environment variable or use the default 1B instruct model.
# For LLaMA-2-13B use "meta-llama/Llama-2-13b-hf" (requires access approval).
# Lightweight alternative (no approval needed): "meta-llama/Llama-3.2-1B-Instruct"
MODEL_ID = os.getenv("MODEL_ID", "meta-llama/Llama-3.2-1B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# LLaMA 沒有內建 pad_token，設為 eos_token 或 id=2
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"  # causal LM 訓練時必須 right-padding
print(tokenizer)

## Step 4  資料預處理：apply_chat_template

### 為何使用 apply_chat_template？

`apply_chat_template` 把 prompt 組裝邏輯封裝在 tokenizer 的 `chat_template` 字串（Jinja2 模板）裡，一次定義、到處使用，且跨模型可攜：

- LLaMA-3 的 `<|start_header_id|>` 格式、ChatGLM 的 `build_chat_input()`、Qwen 的自訂格式 — 統一由各自的 tokenizer 管理，切換模型不需改動訓練/推論程式碼。
- 訓練與推論使用完全相同的模板，確保格式一致性。
- 2026 的多模態訊息（image token、audio token）同樣走 `apply_chat_template` 機制，設計上具備可延伸性。

### SFTTrainer 的 response-only 遮罩

訓練 SFT 時只有 response 部分的 token 應貢獻 loss，instruction 部分用 `-100` 標籤遮罩屏蔽。  
底層原理：
```python
labels = [-100] * len(instruction_ids) + response_ids + [eos_id]
```
SFTTrainer 配合 `DataCollatorForCompletionOnlyLM` 可自動完成這件事，不再需要手刻。

In [ ]:
def build_messages(example: dict) -> list[dict]:
    """Convert alpaca-format example to chat messages list."""
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n\n{example['input'].strip()}"
    return [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]


def formatting_func(example: dict) -> str:
    """Format a single example into a chat-template string for SFTTrainer."""
    messages = build_messages(example)
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # 訓練時不加 generation prompt，完整包含 response
    )


# 驗證格式
print(formatting_func(ds[0]))

In [ ]:
# 底層手刻版（對照教學用）— 說明 -100 遮罩原理
# SFTTrainer 在生產環境已替你做好這一切，這裡只是教學對照

MAX_LENGTH = 512  # LLaMA tokenizer 對中文一字多 token，需要比英文更長

def process_func_manual(example: dict) -> dict:
    """
    Manually build input_ids, attention_mask, and labels with -100 masking.
    This is the low-level version for pedagogical comparison only.
    SFTTrainer handles this automatically in production.
    """
    messages = build_messages(example)

    # 只含 instruction（不含 response）的 prompt
    instruction_text = tokenizer.apply_chat_template(
        messages[:-1],  # 只有 user turn
        tokenize=False,
        add_generation_prompt=True,  # 推論端格式：在末尾加 assistant 開頭
    )
    # 完整 prompt（含 response）
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    instruction_ids = tokenizer(instruction_text, add_special_tokens=False)["input_ids"]
    full_ids       = tokenizer(full_text,        add_special_tokens=False)["input_ids"]

    # response token ids = full - instruction
    response_ids = full_ids[len(instruction_ids):]

    input_ids      = full_ids + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    # -100 遮罩：instruction 部分不計 loss，只算 response + eos
    labels         = [-100] * len(instruction_ids) + response_ids + [tokenizer.eos_token_id]

    # 截斷
    input_ids      = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    labels         = labels[:MAX_LENGTH]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


# 驗證：labels 中非 -100 的部分應等於 response 文字
sample = process_func_manual(ds[0])
response_tokens = [t for t in sample["labels"] if t != -100]
print("Response decoded:", tokenizer.decode(response_tokens, skip_special_tokens=True)[:120])

## Step 5  建立模型：BitsAndBytesConfig 量化

### 4-bit / 8-bit / 16-bit 何時用？

| 精度 | VRAM 節省 | 訓練速度 | 適合場景 |
|------|-----------|----------|----------|
| fp32 | 基準 | 最慢 | 精度敏感的研究實驗 |
| bf16 | -50% | 快 | 現代 GPU（A100/H100/RTX30+）標準訓練精度 |
| fp16 | -50% | 快 | 較舊 GPU；需注意數值穩定性（loss scale） |
| int8 | -75% | 中 | 推論；訓練時搭配 LoRA |
| **nf4** | **-87.5%** | **慢（dequant overhead）** | **QLoRA 訓練；消費級 GPU** |

### NF4 vs FP4

- **FP4**：直接截斷 float，分佈均勻
- **NF4（Normal Float 4）**：依照正態分佈的分位數設計量化邊界，讓最常出現的數值區間（接近零）獲得更高精度。實驗顯示 NF4 在相同 bit 數下 perplexity 更低。

### Double Quantization 成本效益

`bnb_4bit_use_double_quant=True` 對「量化常數」本身再做一次 8-bit 量化，每個參數額外節省約 0.37 bit（整體模型再省 ~3-4% VRAM），推論速度幾乎不受影響，訓練速度僅輕微降低。

### 為何 bf16 優於 fp16？

- **bf16**（Brain Float 16）保留與 fp32 相同的 8-bit 指數範圍，僅縮短尾數（mantissa）。這意味著它有 fp32 的數值範圍，不需要 loss scaling，在現代 GPU（Ampere+）上也有硬體原生支援。
- **fp16** 指數位只有 5-bit，容易在訓練中出現 gradient overflow，需要額外的 loss scaling 機制。
- **compute_dtype 與儲存 dtype 的分離**：模型用 NF4 儲存（省 VRAM），但 forward/backward pass 時動態 dequant 到 `bnb_4bit_compute_dtype`（bf16）進行矩陣乘法，確保計算精度。

### 正確的 PEFT 前置順序

```
1. BitsAndBytesConfig  →  量化設定
2. AutoModelForCausalLM.from_pretrained(quantization_config=...)  →  4-bit 載入
3. prepare_model_for_kbit_training(model)  →  為 kbit 訓練凍結 & 轉換 LayerNorm
4. get_peft_model(model, lora_config)  →  注入 LoRA adapter
```

步驟 3 若省略，`gradient_checkpointing` 開啟時會報 `RuntimeError: element 0 of tensors does not require grad`，或 LayerNorm 在量化精度下累積梯度造成訓練不穩定。

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NF4 > FP4 在正態分佈權重上
    bnb_4bit_compute_dtype=torch.bfloat16,  # 計算精度：bf16 無需 loss scale
    bnb_4bit_use_double_quant=True,         # double quant：每參數再省 ~0.37 bit
)

# device_map='auto'：Accelerate 自動分配到可用的 GPU/CPU/disk
# torch_dtype=torch.bfloat16：非量化層（LayerNorm 等）的 dtype
# use_safetensors=True：safetensors 格式，比 pickle 更安全且載入更快
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
print(model)

In [ ]:
# 檢查各層的精度，確認量化正確套用
for name, param in list(model.named_parameters())[:10]:
    print(f"{name:<60}  shape={str(param.shape):<20}  dtype={param.dtype}")

## Step 6  PEFT 準備與 LoRA 設定

### prepare_model_for_kbit_training 做了什麼？

1. **凍結量化層**：除 LoRA adapter 之外的所有參數設為 `requires_grad=False`
2. **轉換 LayerNorm 到 fp32**：量化層本身不可訓練，但 LayerNorm 在高精度下累積更穩定
3. **啟用 input_require_grads**：配合 gradient checkpointing 讓梯度能正確回傳

若省略這一步，常見症狀：
- `gradient_checkpointing=True` 時報 `RuntimeError: element 0 of tensors does not require grad`
- 訓練 loss 不收斂（LayerNorm 在錯誤精度下更新）

In [ ]:
# Step 6a — 為 kbit 訓練準備模型（舊版常被省略！）
# 必須在 get_peft_model 之前呼叫
model = prepare_model_for_kbit_training(model)

# Step 6b — LoRA 設定
# r=8：低秩矩陣的秩（rank）。越大能力越強，VRAM 越多
# lora_alpha=32：縮放係數（通常 2*r 是合理起點）
# target_modules：要注入 LoRA 的層名稱（attention 投影層）
# lora_dropout=0.05：防止 adapter 過擬合
# bias='none'：不訓練 bias（節省參數）
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 預期輸出類似：trainable params: 3,407,872 || all params: 1,240,222,720 || trainable%: 0.27

## Step 7  訓練設定：SFTConfig

### SFTTrainer 的優勢

`SFTTrainer` 在 `Trainer` 之上提供以下能力：

- **Response-only 標籤遮罩**：自動遮蔽 instruction token 的 loss，無需手刻 `-100` 陣列
- **formatting_func**：接收 example dict，回傳格式化字串，與 `apply_chat_template` 自然整合
- **Packing**：`packing=True` 可把短序列打包成長序列，減少 padding 浪費、提升吞吐量
- **SFTConfig 繼承 TrainingArguments**：所有標準訓練參數可直接使用，無需切換類別

### 重要訓練參數說明

- `bf16=True`：在支援的 GPU 上使用 bf16 混合精度
- `gradient_checkpointing=True`：以重算換 VRAM，訓練長序列的必要選項
- `optim='paged_adamw_8bit'`：QLoRA 論文推薦的 optimizer，8-bit 分頁版節省 VRAM
- `warmup_ratio=0.03`：前 3% step 線性 warmup，穩定訓練初期的大梯度
- `lr_scheduler_type='cosine'`：cosine decay 讓學習率平滑下降
- `gradient_accumulation_steps=16`：有效 batch size = per_device_batch × grad_accum

> **有效 batch size 計算**：`effective_batch = per_device_train_batch_size × gradient_accumulation_steps × num_gpus`  
> 本例：1 × 16 × 1 = 16（等效 global batch size 16）

In [ ]:
sft_config = SFTConfig(
    output_dir="./chatbot-qlora",

    # --- batch & gradient ---
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,   # effective batch = 1 × 16 = 16

    # --- 訓練長度 ---
    num_train_epochs=1,
    max_steps=-1,                     # -1 = 由 epoch 決定
    max_seq_length=512,               # SFTTrainer 控制最大序列長度

    # --- 精度 ---
    bf16=True,                        # bf16（無 loss scaling）
    gradient_checkpointing=True,      # 以重算換 VRAM

    # --- optimizer & scheduler ---
    learning_rate=2e-4,
    optim="paged_adamw_8bit",         # QLoRA 推薦；32bit 更穩但多用 VRAM
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.001,
    max_grad_norm=1.0,

    # --- logging & saving ---
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    save_safetensors=True,            # 以 safetensors 格式儲存 checkpoint

    # --- 可重現性 ---
    seed=42,

    # --- SFT 專屬 ---
    dataset_text_field=None,          # 使用 formatting_func
    packing=False,                    # True 可加速但需確認 chat template 相容
)

print(sft_config)

## Step 8  建立 SFTTrainer 與開始訓練

### SFTTrainer 的 formatting_func 路徑

```python
SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,   # processing_class 取代舊版的 tokenizer= 參數
    formatting_func=formatting_func,
    peft_config=lora_config,      # 也可以不傳（已在 Step 6 apply）
)
```

`formatting_func` 每次接收一個 example dict，回傳格式化後的字串；SFTTrainer 再呼叫 tokenizer 將其 tokenize 並自動做 response-only masking（若有 `DataCollatorForCompletionOnlyLM`）。

> **輕量替代**：若 VRAM 不足，可換用 `meta-llama/Llama-3.2-1B-Instruct`（1B 參數，約 1.5 GB VRAM@4bit）

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds.select(range(6000)),  # 快速實驗用 6000 筆
    processing_class=tokenizer,            # processing_class 為 trl>=0.12 新參數名
    formatting_func=formatting_func,       # apply_chat_template 封裝
    # peft_config=lora_config,             # 若尚未 get_peft_model，在此傳入也可
)

trainer.train()

## Step 9  模型推理（訓練後驗證）

推論時同樣用 `apply_chat_template` 組 prompt，確保與訓練時格式完全一致。  
這是「訓練/推論一致性」最關鍵的一步：若推論格式與訓練不同，模型回應品質會明顯下降。

In [ ]:
model.eval()

def generate_response(instruction: str, input_text: str = "") -> str:
    """Generate response using the same chat template as training."""
    user_content = instruction
    if input_text.strip():
        user_content = f"{instruction}\n\n{input_text.strip()}"

    messages = [{"role": "user", "content": user_content}]

    # add_generation_prompt=True：在末尾加上 assistant 開頭，提示模型開始生成
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # 只 decode 新生成的 token（去除 input prompt）
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


result = generate_response("你好，請介紹一下自己")
print(result)

In [ ]:
# 多組測試
test_cases = [
    ("請給我三個保持健康的建議", ""),
    ("把下列句子翻譯成英文", "人工智慧正在改變世界"),
    ("請解釋什麼是量子纏結", ""),
]

for instruction, inp in test_cases:
    print(f"指令：{instruction}")
    if inp:
        print(f"輸入：{inp}")
    print(f"回應：{generate_response(instruction, inp)}")
    print("-" * 60)

## Step 10  合併 LoRA Adapter 並儲存

`merge_and_unload()` 把 LoRA 的低秩矩陣合併回原始權重，移除 PEFT adapter wrapper，得到一個標準的 `AutoModelForCausalLM`，方便後續部署或推送至 HF Hub。

注意：合併後模型會回到全精度（依 `torch_dtype`），VRAM 使用量會上升。若 VRAM 不足，可只儲存 adapter（`model.save_pretrained()`）不合併。

In [ ]:
# 合併 LoRA adapter 到基礎模型
merged_model = model.merge_and_unload()

# 儲存合併後的模型（safe_serialization=True：safetensors 格式，非 pickle）
OUTPUT_DIR = "./chatbot-qlora-merged"
merged_model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Merged model saved to {OUTPUT_DIR}")

In [ ]:
# 選擇性：推送至 HuggingFace Hub
# 需先執行 huggingface-cli login 或設定 HF_TOKEN 環境變數
#
# HUB_MODEL_ID = "your-username/llama3-1b-alpaca-zh-qlora"
# merged_model.push_to_hub(
#     HUB_MODEL_ID,
#     safe_serialization=True,
#     private=True,
# )
# tokenizer.push_to_hub(HUB_MODEL_ID)
# print(f"Model pushed to https://huggingface.co/{HUB_MODEL_ID}")

## 小結

### 核心概念回顧

1. **NF4 量化**：依正態分佈分位數設計邊界，在 4-bit 下比 FP4 更低 perplexity
2. **Double Quantization**：對量化常數再量化，額外省 ~3-4% VRAM，幾乎無速度損失
3. **bf16 計算精度**：有 fp32 的指數範圍，無 loss scaling 需求，現代 GPU 首選
4. **PEFT 正確順序**：`prepare_model_for_kbit_training()` 在 `get_peft_model()` 之前
5. **apply_chat_template**：訓練/推論使用同一模板，跨模型可攜，多模態可延伸

### 練習題

1. 把 `bnb_4bit_use_double_quant` 設為 `False`，比較訓練前後的 VRAM 使用量差異
2. 把 `bnb_4bit_quant_type` 從 `"nf4"` 改為 `"fp4"`，觀察 loss 曲線是否有差異
3. 嘗試在 `lora_config` 中加入 `target_modules=["q_proj", "v_proj"]`（去掉 k/o），比較可訓練參數數量與最終效果
4. 把 `packing=True` 傳入 `SFTConfig`，觀察訓練速度的變化，並思考 packing 對 chat template 的潛在問題
5. （進階）在 `process_func_manual` 中加入斷言，確認 `-100` 遮罩後 decode 出的文字與 `example["output"]` 一致